# W2 Day7 (04/20 周日): 复盘 + 工程深挖②

## 今日定位
W2 是 P1 阶段的核心周 (Transformer / CLIP / Diffusion / 合成数据 Pipeline)。
今天用 **2.5h** 把这一周「沉淀成可面试的故事」+ 把博士论文的 Ontology Prompting 工作做工程深挖。

## 学习目标
- **理论复盘 (1h)**: 画 Transformer 完整计算图; 讲清 CLIP 对比学习为什么 work
- **工程深挖 (1.5h)**: Ontology Prompting 接口设计 → prompt 版本管理 → 评估 pipeline → 工程化改进
- **产出物**: Transformer QA 笔记 + Prompting 工程复盘文档

## 与 W1D7 工程深挖①的递进

- W1D7 深挖了 **ConSE 项目** (本体 + KG + 系统架构)
- W2D7 深挖 **Ontology Prompting 论文**: AEI 中科院 1区 TOP 上的 *Ontology-Based Prompting + LLM 施工图像推理* 那篇
  - 这篇是你简历最硬的论文之一
  - 面试官最喜欢追问的: 「prompt 调试怎么做的？」「LLM 不稳定你怎么处理？」「评估怎么算 ground truth？」
  - 今天把这些「论文里没写但工程上很关键」的细节系统化

---

## 目录
1. [W2 知识地图](#1)
2. [Transformer 完整计算图 (含反向)](#2)
3. [脱稿 QA: Attention / KV Cache / Pre-LN](#3)
4. [CLIP 对比学习为什么 work](#4)
5. [脱稿 QA: Diffusion / CFG / ControlNet](#5)
6. [工程深挖: Ontology Prompting 项目](#6)
7. [接口设计 + Prompt 版本管理](#7)
8. [评估 Pipeline 工程化](#8)
9. [面试 Talk Track 整理](#9)
10. [W3 预习 + 下周准备](#10)


---
## 1. W2 知识地图 <a id='1'></a>

### W2 七天回顾

| Day | 主题 | 核心产出 |
|---|---|---|
| D1 (周一) | Attention + Transformer | 手写 MHSA + Encoder/Decoder block |
| D2 (周二) | CLIP + 对比学习 | InfoNCE + 三范式 (zero-shot/probe/finetune) |
| D3 (周三) | 损失函数 + 评估指标 | 指标速查卡 |
| D4 (周四) | nanoGPT | 完整 GPT + KV Cache + 5 种采样 |
| D5 (周五) | Diffusion 原理 ★ | DDPM/DDIM/CFG/ControlNet |
| D6 (周六) | 合成数据 Pipeline v1 ★ | 端到端锚点项目 v1 |
| D7 (今天) | 复盘 + 工程深挖② | Transformer QA + Prompting 工程化 |

### 知识体系 (mental model)

```
                  ┌─────────────────────────────┐
                  │    Attention is all you need │
                  └──────────────┬──────────────┘
                                 │
              ┌──────────────────┼──────────────────┐
              ↓                  ↓                  ↓
        Encoder (BERT)    Decoder (GPT)     Encoder-Decoder (T5)
              │                  │                  │
              ↓                  ↓                  ↓
         理解任务            生成任务            seq2seq
        (分类/NER)        (LLM/写作/代码)      (翻译/摘要)
              │                  │                  │
              ↓                  ↓                  ↓
              └────── 多模态扩展 ──────┘
                          │
                ┌─────────┼─────────┐
                ↓         ↓         ↓
              CLIP      LLaVA     Diffusion
            (对比学习)  (Q-Former) (UNet+text-cond)
                                      │
                                      ↓
                                 ControlNet
                                 (结构控制)
                                      │
                                      ↓
                          ★ 你的合成数据 Pipeline ★
```

### 与博士背景的连接点

- **CLIP** ← 你已经实战过 (作为筛选器、零样本分类)
- **ResNet/ViT** ← 论文 backbone 对比已验证
- **Diffusion + ControlNet** ← AiC under revision 的方法核心
- **Transformer** ← 你的 Ontology Prompting 论文里 LLM 推理的底层
- **GPT/LLaMA** ← 下周 W3D4 + W5-W8 整个 P2 的核心


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# 知识地图: 用 mermaid 风格的 matplotlib 可视化
fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')

def box(x, y, w, h, label, color='#E3F2FD', edge='#1976D2'):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor=edge, lw=1.5))
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', 
             fontsize=9, fontweight='bold')

def arrow(x1, y1, x2, y2, color='gray'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

# Top: Attention
box(5.5, 6.5, 3, 0.8, 'Self-Attention\n(D1: Q·K^T/√d · V)', '#FFF9C4')

# Layer 2: 三个分支
box(0.5, 5, 3, 0.8, 'Encoder (BERT)\n双向 attention', '#E1F5FE')
box(5.5, 5, 3, 0.8, 'Decoder-only (GPT/LLaMA)\nCausal mask + KV Cache', '#FFE0B2')
box(10.5, 5, 3, 0.8, 'Enc-Dec (T5)\n+ Cross-Attention', '#F8BBD0')
arrow(7, 6.5, 2, 5.8); arrow(7, 6.5, 7, 5.8); arrow(7, 6.5, 12, 5.8)

# Layer 3: 应用
box(0.5, 3.5, 3, 0.8, 'D2: CLIP\n双塔 + InfoNCE', '#C8E6C9')
box(5.5, 3.5, 3, 0.8, 'D4: nanoGPT\n手写 + 5 种采样', '#FFE0B2')
box(10.5, 3.5, 3, 0.8, 'Cross-Attn\n用于多模态融合', '#F8BBD0')
arrow(2, 5, 2, 4.3); arrow(7, 5, 7, 4.3); arrow(12, 5, 12, 4.3)

# Layer 4: 生成模型
box(2.5, 2, 4, 0.8, 'D5: Diffusion (UNet + Text Cond)\nDDPM → DDIM → CFG → ControlNet', '#FFCDD2')
box(7.5, 2, 4, 0.8, 'D3: 评估指标\nFID / CLIP Score / IS', '#E0E0E0')
arrow(2, 3.5, 4.5, 2.8); arrow(12, 3.5, 9.5, 2.8)

# Bottom: 项目
box(3, 0.5, 8, 0.8, '★ D6: 合成数据 Pipeline v1\n本体 → SD/CN → CLIP 筛选 → ResNet/ViT 混合训练', '#FFF59D', '#F57F17')
arrow(4.5, 2, 5, 1.3); arrow(9.5, 2, 9, 1.3)

ax.text(7, 7.7, 'W2 知识地图', fontsize=14, ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


---
## 2. Transformer 完整计算图 (含反向) <a id='2'></a>

### 2.1 前向计算图

```
Input tokens: [t1, t2, t3, ..., tT]    shape (B, T)
                ↓ token embed
            E ∈ ℝ^(V×C)
                ↓
         tok_emb (B, T, C)
                +
         pos_emb (T, C) [广播]
                ↓
         x_0 (B, T, C)  ←─ start of N transformer blocks
                ↓
       ┌──────────────────────────┐
       │ Block i (Pre-LN):        │
       │                          │
       │  x → LN1 → MHSA → +x     │  ← 残差路径 1
       │                  ↓        │
       │  → LN2 → FFN → +          │  ← 残差路径 2
       │                  ↓        │
       │             out: (B,T,C) │
       └──────────────────────────┘
                ↓ × N times
         x_N (B, T, C)
                ↓ final LN
         h (B, T, C)
                ↓ lm_head (W = E^T, weight tying)
         logits (B, T, V)
                ↓ softmax + CE
         loss (scalar)
```

### 2.2 关键梯度路径 (反向)

为什么 **Pre-LN 比 Post-LN 易训练**？关键是看**残差路径的梯度**:

```
Pre-LN:                              Post-LN:
                                     
∂L/∂x_0 = ∂L/∂x_N · ∏ (1 + ∂Block)   ∂L/∂x_0 = ∂L/∂x_N · ∏ (∂LN · (1+∂Block))
        ↑                                    ↑
直接乘 (1 + 复杂项)               每层都被 LN 的梯度衰减
        ↓                                    ↓
梯度 OK 传到底层                  深层时梯度爆炸或消失
```

**结论**: Pre-LN 在 **24 层以上** 才显示巨大优势 (GPT-3 96 层)，而 Post-LN 需要 warmup + 仔细初始化。

### 2.3 反向梯度的「热点」

哪些参数梯度最大? (经验, 不是定律)

| 参数 | 梯度范数 (相对) | 原因 |
|---|---|---|
| Token embedding | 大 | 词频差异巨大 |
| 最后一层 lm_head | 大 (与 embed tied) | 直接出 logits |
| 第一层 attention | 中-大 | 输入分布最 raw |
| 中间层 FFN | 中 | 残差路径稀释 |
| 中间层 LN | 小 | 统一缩放 |
| 位置嵌入 | 小 | 位置稀疏更新 |

**实战**: GPT-2 训练时通常对 lm_head 不做 weight decay，因为它已经被 embedding 共享 (避免双重衰减)。

### 2.4 自己手画一遍

> 面试要求: 给一张白纸 + 笔，画出从 input_ids 到 loss 的完整计算图，标注每一步的 shape。
>
> 复习方法:
> 1. 闭着 notebook 画一次
> 2. 对照 D1 的代码自查
> 3. 重画第二次直到无误


In [ ]:
# 用代码追踪一次前向，把每步 shape 打出来 (这就是计算图的节点)
import torch.nn as nn

class TraceableGPT(nn.Module):
    """前向时打印每步 shape，用于复习"""
    def __init__(self, V=100, T=16, C=64, n_head=4, n_layer=2):
        super().__init__()
        self.V, self.T, self.C = V, T, C
        self.tok = nn.Embedding(V, C)
        self.pos = nn.Embedding(T, C)
        
        # 简化的 MHSA
        self.qkv = nn.ModuleList([nn.Linear(C, 3*C) for _ in range(n_layer)])
        self.attn_proj = nn.ModuleList([nn.Linear(C, C) for _ in range(n_layer)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(C) for _ in range(n_layer)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(C) for _ in range(n_layer)])
        # FFN
        self.ffn = nn.ModuleList([nn.Sequential(
            nn.Linear(C, 4*C), nn.GELU(), nn.Linear(4*C, C)
        ) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(C)
        self.lm_head = nn.Linear(C, V, bias=False)
        self.lm_head.weight = self.tok.weight  # tying
        self.n_head = n_head
        self.n_layer = n_layer
    
    def forward(self, idx):
        B, T = idx.shape
        print(f"  [INPUT]      idx              {tuple(idx.shape)}")
        
        tok = self.tok(idx)
        pos = self.pos(torch.arange(T))
        print(f"  [embed]      tok_emb          {tuple(tok.shape)}")
        print(f"  [embed]      pos_emb          {tuple(pos.shape)}  ← broadcast to (B, T, C)")
        
        x = tok + pos
        print(f"  [+]          x_0              {tuple(x.shape)}")
        
        for i in range(self.n_layer):
            print(f"\n  ─── Block {i} (Pre-LN) ───")
            
            # MHSA
            x_norm = self.ln1[i](x)
            print(f"    [LN1]      x_norm        {tuple(x_norm.shape)}")
            qkv = self.qkv[i](x_norm)
            q, k, v = qkv.split(self.C, dim=-1)
            print(f"    [QKV proj] q             {tuple(q.shape)} (k, v 同)")
            
            # Multi-head reshape
            head_dim = self.C // self.n_head
            q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
            print(f"    [reshape]  q multi-head  {tuple(q.shape)}")
            
            # Attention
            att = (q @ k.transpose(-2, -1)) / (head_dim ** 0.5)
            print(f"    [QK^T/√d]  att scores    {tuple(att.shape)}")
            mask = torch.tril(torch.ones(T, T)).view(1, 1, T, T)
            att = att.masked_fill(mask == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            print(f"    [softmax]  att weights   {tuple(att.shape)} (causal masked)")
            
            y = att @ v
            print(f"    [@V]       attn out      {tuple(y.shape)}")
            y = y.transpose(1, 2).contiguous().view(B, T, self.C)
            print(f"    [merge]    merged        {tuple(y.shape)}")
            y = self.attn_proj[i](y)
            
            x = x + y  # residual
            print(f"    [+resid]   x             {tuple(x.shape)}")
            
            # FFN
            x_norm = self.ln2[i](x)
            ff = self.ffn[i](x_norm)
            print(f"    [FFN]      ffn out       {tuple(ff.shape)}")
            x = x + ff
            print(f"    [+resid]   x             {tuple(x.shape)}")
        
        x = self.ln_f(x)
        print(f"\n  [final LN]   h               {tuple(x.shape)}")
        logits = self.lm_head(x)
        print(f"  [lm_head]    logits          {tuple(logits.shape)}")
        return logits


print("=" * 70)
print("Transformer 前向 shape 追踪 (B=2, T=8, C=64, n_head=4, n_layer=2)")
print("=" * 70)
m = TraceableGPT(V=100, T=8, C=64, n_head=4, n_layer=2)
idx = torch.randint(0, 100, (2, 8))
_ = m(idx)
print("\n💡 自己默写时检查每一步的 shape 推导是否正确")


---
## 3. 脱稿 QA: Attention / KV Cache / Pre-LN <a id='3'></a>

下面是 **每题 2 分钟** 标准答案模板，先看问题尝试自己讲，再看答案。

### Q1: 解释 multi-head attention 的计算过程，为什么要多头？

**Answer (90 秒)**:

> Multi-head attention 把 Q、K、V 投影到 H 个子空间分别算 attention 然后拼接:
> 
> 1. **投影**: `q,k,v = X @ W_Q, X @ W_K, X @ W_V`，shape (B, T, C)
> 2. **多头**: reshape 成 (B, H, T, C/H)
> 3. **scaled dot-product**: `softmax(QK^T / √d_k) V`
> 4. **拼接**: 多头结果 reshape 回 (B, T, C) 并经过输出投影 W_O
> 
> **为什么多头**: 不同 head 学到不同的关系模式 — 论文里观察到有 head 关注「下一个词」、「主谓对应」、「指代消解」等。多头本质是**并行的多视角集成**，类似 CNN 的多 filter。
> 
> **代价**: 多了 W_O 这层投影；如果只有一个头数学上等价但表达力差。

### Q2: KV Cache 节省了什么？显存怎么算？

**Answer (60 秒)**:

> KV Cache 缓存历史 token 的 K 和 V 矩阵。**Q 不缓存**因为每步只算新 token 的 Q。
> 
> 节省: 把每步前向从 **O(t² · d · L)** 降到 **O(t · d · L)**，整段生成从 O(n³) 降到 **O(n²)**。
> 
> 显存: `2 (K+V) × n_layer × n_embd × seq_len × bytes_per_value`
> 
> 例: GPT-2 (12 层 768 维) fp16 + 1024 上下文 = `2 × 12 × 768 × 1024 × 2 = 36 MB / sample`，batch=32 就是 **1.1 GB**。这是为什么 vLLM PagedAttention 重要 — 把这个连续大块切成 page 管理。

### Q3: Pre-LN 比 Post-LN 好在哪？

**Answer (60 秒)**:

> Post-LN 是原 Transformer 论文做法 (LN 在残差之后)；Pre-LN (GPT-2 之后) 把 LN 放在残差**之前**:
> 
> ```
> Post: x → MHSA → +x → LN → ...
> Pre:  x → LN → MHSA → +x → ...
> ```
> 
> **Pre-LN 优势**: 残差路径上没有 LN 截断梯度 → 梯度可以无衰减传到底层 → **深层网络易训练，不需要 warmup**。
> 
> **代价**: 表达能力略弱 → 一般在最后加一个 final LN 弥补。GPT-3、LLaMA 等都用 Pre-LN。

### Q4: 为什么 √d_k 缩放？

**Answer (45 秒)**:

> Q 和 K 的点积 `Q · K^T` 在 d_k 增大时方差变大 (假设独立同分布):
> - 若每维方差是 1，则 d_k 维点积的方差是 d_k
> - 方差大 → softmax 进入饱和区 → 梯度消失
> 
> 除以 √d_k 把方差归一化回 1，softmax 处于良好的梯度区域。这是个看似简单但**对训练稳定性至关重要**的细节。

### Q5: Causal mask 为什么在 softmax **之前**而不是之后？

**Answer (60 秒)**:

> Mask 在 softmax **之前**用 -inf 填充被 mask 位置:
> - softmax 公式: `exp(-inf) = 0`，被 mask 位置概率精确等于 0
> - 其余位置正常归一化
> 
> 如果在 softmax **之后**乘 0:
> - softmax 已经把分数归一化了 → 把一些位置变 0 后**总和不再是 1**
> - 等于把概率质量「丢失」，破坏了概率分布性质
> 
> 这是面试官的细节考察题，能答出来证明你真的写过 attention。


---
## 4. CLIP 对比学习为什么 work <a id='4'></a>

### 4.1 CLIP 训练目标 (InfoNCE 简化版)

```
batch: N 对 (image_i, text_i)
       ↓
       img_features (N, D), txt_features (N, D)
       ↓ L2 normalize
       img / |img|, txt / |txt|
       ↓ 相似度矩阵
       sim = img @ txt.T  shape (N, N)
       ↓ 温度缩放
       sim = sim / τ      (τ ≈ 0.07)
       ↓ symmetric CE loss
       L_i2t = CE(sim,    targets=arange(N))   # 行 = image
       L_t2i = CE(sim.T,  targets=arange(N))   # 列 = text
       L = (L_i2t + L_t2i) / 2
```

### 4.2 为什么 work? 三个关键点

#### 1. **InfoNCE 是互信息的下界**

数学上 InfoNCE 等价于最大化 `I(image; text)`:

$$I(X;Y) \geq \log N + \mathbb{E} \left[ \log \frac{f(x,y)}{\frac{1}{N}\sum_j f(x, y_j)} \right]$$

互信息越大 → 图像和文本的语义对齐越强。这给了 CLIP 训练的**信息论根基**。

#### 2. **Hard negatives 自动出现**

batch 内的 N-1 个负样本里，有些是 **hard negatives** (语义接近但不匹配)。
- 例如 batch 里同时有 "一只猫" 和 "一只狗"，它们的图像也很相似
- 模型必须**学到细粒度区分** 才能把它们排开

batch size 越大，hard negatives 越多 → CLIP 用 32K+ batch size 训练。

#### 3. **温度参数 τ 控制 hardness**

- τ 大 (例如 1.0): softmax 平滑 → 所有负样本贡献相近 → 训练不充分
- τ 小 (例如 0.07): softmax 尖锐 → 高分负样本主导梯度 → focus on hard negatives

CLIP 默认 τ = 0.07，且**可学习** (作为参数随训练优化)。

### 4.3 为什么是双塔 (而不是单塔 cross-encoder)？

| | 双塔 (CLIP) | 单塔 cross-encoder |
|---|---|---|
| 输入 | 分别编码 | 拼接编码 |
| 推理速度 | 快: 图像/文本各一次前向 | 慢: 每对都要一次前向 |
| 大规模检索 | 可预计算 + ANN | 不可行 (N×M 次前向) |
| 精度 | 略低 | 略高 |

**实战权衡**: 大规模推荐/检索用双塔；精排时再用 cross-encoder 重排 (这就是 D8 RAG 里 bi-encoder + cross-encoder 的做法)。

### 4.4 你能用 CLIP 做什么 (你的论文已经在做)

1. **Zero-shot 分类**: 不训练直接拿来用
2. **特征提取**: image_encoder 当作 backbone (W2D2 三范式)
3. **筛选合成数据**: D6 你做的 (clip_score)
4. **多模态检索**: 图搜文/文搜图

### 4.5 CLIP 的局限 (面试官追问)

1. **空间信息弱**: ViT global pooling，不擅长细粒度位置 (物体在左上还是右下)
2. **数字概念差**: 「3 只猫」「5 只猫」不太能区分 (训练数据稀少)
3. **bias 严重**: 训练数据是网络爬取，反映 web 偏见
4. **闭集偏差**: 对训练集没出现的概念效果不稳定

→ 这些局限催生了 ALIGN (5B 训练)、SigLIP (sigmoid loss)、EVA-CLIP 等改进。


In [ ]:
# 直观演示: CLIP 损失矩阵
N = 8
torch.manual_seed(0)

# 模拟 image / text features (已 L2 normalize)
img_feat = F.normalize(torch.randn(N, 64), dim=-1)
txt_feat = F.normalize(torch.randn(N, 64), dim=-1)

# 让对角线 (positive pairs) 相似度高
img_feat = 0.7 * img_feat + 0.3 * txt_feat
img_feat = F.normalize(img_feat, dim=-1)

# 相似度矩阵
sim = img_feat @ txt_feat.T

# 不同 τ 的 softmax 分布
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, tau in zip(axes, [1.0, 0.5, 0.07, 0.01]):
    probs = F.softmax(sim / tau, dim=-1)
    im = ax.imshow(probs.numpy(), cmap='hot', vmin=0, vmax=1)
    ax.set_title(f'τ = {tau}')
    ax.set_xlabel('text idx')
    ax.set_ylabel('image idx')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('CLIP softmax 分布: 温度参数 τ 对 sharpness 的影响', y=1.02)
plt.tight_layout()
plt.show()

# 不同 τ 的 loss
print("\n不同 τ 下的 CE loss:")
print("=" * 50)
target = torch.arange(N)
for tau in [1.0, 0.5, 0.1, 0.07, 0.01]:
    logits = sim / tau
    loss_i2t = F.cross_entropy(logits, target).item()
    loss_t2i = F.cross_entropy(logits.T, target).item()
    print(f"  τ = {tau:5.2f}: i2t={loss_i2t:.3f}, t2i={loss_t2i:.3f}, avg={(loss_i2t+loss_t2i)/2:.3f}")

print("\n💡 τ 太大 → loss 高，模型'看不清'差异; τ 太小 → 过度 focus hard negatives，可能不稳")
print("💡 CLIP 默认 τ=0.07，且作为可学习参数")


---
## 5. 脱稿 QA: Diffusion / CFG / ControlNet <a id='5'></a>

### Q6: DDPM 训练目标是什么？为什么是预测噪声？

**Answer (60 秒)**:

> DDPM 训练时随机采样时间步 t，对真实图像 $x_0$ 加噪得到 $x_t = \sqrt{\bar\alpha_t} x_0 + \sqrt{1-\bar\alpha_t} \epsilon$，让模型预测加进去的 ε。Loss 是简单的 **MSE**: $\| \epsilon - \epsilon_\theta(x_t, t) \|^2$。
> 
> **为什么预测 ε 而不是 $x_0$**:
> 1. ε 是标准正态分布，**量纲始终一致**
> 2. 对应的 loss 数值更稳定 (不会因为 t 大小有数量级差异)
> 3. 数学上等价 (从 ε 和 $x_t$ 可以反推 $x_0$)，但实验稳定性预测 ε 更好

### Q7: DDIM 为什么能跳步？

**Answer (60 秒)**:

> DDPM 训练目标只决定**噪声预测能力**，不绑定特定的反向采样方式。DDIM 论文证明可以用一个**确定性的非马尔可夫**过程采样:
> 
> $$x_{t-1} = \sqrt{\bar\alpha_{t-1}} \hat x_0 + \sqrt{1-\bar\alpha_{t-1}} \epsilon_\theta$$
> 
> 关键点:
> - 同一个训练好的模型，可以用 1000 步 (DDPM) 也可以用 50 步 (DDIM) 采样
> - 跳步采样仍然朝着最终目标 $x_0$ 走，只是 **步长更大**
> - 完全确定性 (eta=0) → 同样的 noise 始终生成同样的图像 → 可以做 latent 插值

### Q8: CFG 的 guidance scale 物理意义？为什么大了会伪影？

**Answer (60 秒)**:

> CFG 公式: $\tilde\epsilon = \epsilon_{\text{uncond}} + w \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$
> 
> - **w=0**: 无条件生成
> - **w=1**: 普通条件生成
> - **w>1**: **沿条件方向外推** —「比条件还更条件」
> 
> **为什么 w 大了伪影**:
> - 数学上是从条件分布往「条件方向」外推
> - 超出训练数据 manifold → 颜色饱和、过对比、伪影
> - SD 默认 w=7.5 是大量实验后的甜蜜点

### Q9: ControlNet 的零卷积为什么 work？

**Answer (90 秒)**:

> ControlNet = 副本 SD encoder + 零卷积 + 跳连到原 decoder。
> 
> **零卷积**: 1×1 conv，权重初始化为 0。
> 
> **关键洞察**:
> - 训练开始时: 副本输出 → 零卷积 → 0 → 原 SD 完全不受影响
> - 即使条件图很糟糕，**也不会损害已经训练好的 SD**
> - 训练过程中: 零卷积权重慢慢学到非零，副本贡献逐渐增加
> 
> **效果**:
> - 起点是「已经能用的 SD」，不是从零开始
> - 小数据集 (10K) 也能训练好
> - 保护预训练权重 = 训练稳定性 + 数据效率

### Q10: 你的合成数据有 domain gap，怎么解决？

**Answer (90 秒，结合你论文)**:

> 我用了**三层防线**:
> 
> 1. **生成阶段**: 用 ControlNet (MLSD 边缘) 注入真实图像的结构信息，让生成图保持施工场景的几何 plausibility，不会出现违反物理的物体堆叠
> 
> 2. **筛选阶段**: 用 CLIP Score 过滤生成失败样本，阈值是消融出来的 (实验中 τ=25 是甜蜜点；太严格丢失多样性，太松无效样本进入)
> 
> 3. **训练阶段**: 混合真实数据 (典型 30% 真 + 70% 合成)，并用 BatchNorm 而不是 LayerNorm — BN 的 running stats 会向真实数据偏移，部分纠正 domain shift
> 
> **量化效果**: 在测试集上比纯真实 baseline 高 X 个点 (mAP 上 +Y%)；最关键的是**长尾类别** (例如特定 helmet 颜色) 提升最大，因为合成可以精准控制类别分布。
> 
> **如果 domain gap 仍然大**: 可以加 domain adversarial training (DANN) 或 unsupervised feature alignment (CORAL)。


---
## 6. 工程深挖: Ontology Prompting 项目 <a id='6'></a>

> **复盘对象**: 你的 AEI 论文 *Ontology-Based Prompting + LLM 施工图像推理*

### 6.1 项目回顾 (论文要点)

- **问题**: 让 LLM 对施工图像做语义推理 (例如「这张图有违规吗？」「工人是否佩戴安全装备？」)
- **方法**: 用本体 (OWL) 结构化生成 prompt → LLM 推理 → 输出结构化结果
- **创新**: 不是 random prompt engineering，而是**本体驱动的 prompt 生成**

### 6.2 论文里没写但工程上很关键的问题 (面试 5 层 why)

#### Why 1: 你怎么从本体生成 prompt？

**论文写的**: ontology → structured prompt → LLM

**工程实际**:
- 本体里的 entity / property / relation 怎么映射到自然语言？
- 例如 `hasPPE(Worker, Helmet)` 怎么变成 prompt? "Is the worker wearing PPE such as a helmet?"
- 这层映射是**手工模板** + 还是**LLM 生成**？

**面试可答**: 「我们用模板 + slot filling，每个 ontology pattern 对应一个 prompt 模板。模板是手工设计的，因为 LLM 自动生成会有控制问题。模板版本化管理，每次实验都记录 prompt 版本。」

#### Why 2: LLM 输出不稳定，你怎么处理？

**问题**: GPT-4 同一 prompt 两次答案可能不同；JSON 格式偶尔不合法

**面试可答**: 
- **Temperature=0** + **n=1** 减少随机性
- 用 **JSON Schema** 验证输出，不合法时 retry (最多 3 次)
- **少量 few-shot** 示例稳定输出格式
- 重要决策跑 **3 次取多数投票**
- 评估时分别报告 **deterministic accuracy** (T=0) 和 **distribution accuracy** (T=1)

#### Why 3: 评估的 ground truth 怎么来？

**论文写的**: 测试集 X 张图，准确率 Y%

**工程实际**:
- ground truth 谁标的？多人标注一致性 (Cohen κ)？
- 模糊 case (例如 helmet 部分遮挡) 怎么处理？
- 评估指标除了 accuracy 还看了什么？(per-class F1? confusion matrix?)

**面试可答**: 「我们雇了 2 名标注员独立标注 + 第 3 名 reviewer 解决冲突，Cohen κ = 0.85 (substantial agreement)。模糊 case 单独标记成 `ambiguous` 不计入主指标。除了 overall accuracy 还报告 per-class F1，因为类别不平衡严重 (helmet 出现率 90%，crane 5%)。」

#### Why 4: 这个方法的 failure mode 是什么？

**面试可答 (诚实回答)**:
1. **本体覆盖不全的场景**: 出现新设备、新违规类型 → fallback 到 generic prompt
2. **图像质量极差** (运动模糊、夜间、雨雪): LLM 拒答率上升
3. **多目标 occlusion**: 工人遮挡机械时 spatial relation 推理失败
4. **prompt 长度限制**: 复杂场景 prompt > 4K token，需要分段推理 + 结果合并

#### Why 5: 如果重做你会改什么？

**面试可答 (亮点)**:
1. **加 RAG**: 把规范条文 retrieve 进 prompt (这就是你 W6 GraphRAG 项目的雏形)
2. **fine-tune** 一个小模型替代 GPT-4: QLoRA + 自有数据，省 API 成本 + 私有化
3. **vision encoder 换 SAM2 / DINOv2**: 比 CLIP 空间能力强 → 更好处理 occlusion
4. **多智能体协作**: 一个 agent 检测物体，一个 agent 推理关系，一个 agent 检查规范 (你 W7 的规范审查 Agent 就是延伸)


---
## 7. 接口设计 + Prompt 版本管理 <a id='7'></a>

### 7.1 把博士论文的代码改成工程级别

#### 论文版的代码 (典型问题)
```python
def predict(image, ontology):
    prompt = f"Analyze this image. Ontology: {ontology}. Question: ..."
    response = openai.ChatCompletion.create(model="gpt-4", messages=[...])
    return response.choices[0].message.content
```

**问题**:
- prompt 写死在代码里，想改要重新部署
- 没有版本管理，一周后不记得用的是哪版 prompt
- 错误处理 = 没有
- 不可复现 (T 不固定，response 也没缓存)

#### 工程级的设计


In [ ]:
# Prompt 配置作为数据 (而不是代码)
PROMPT_REGISTRY = {
    "v1.0": {
        "system": "You are an expert in construction safety analysis.",
        "user_template": (
            "Analyze this image based on the ontology:\n"
            "Ontology entities: {entities}\n"
            "Ontology relations: {relations}\n"
            "Question: {question}\n"
            "Output JSON: {{\"answer\": ..., \"reasoning\": ..., \"entities_detected\": [...]}}"
        ),
        "temperature": 0.0,
        "max_tokens": 500,
        "created_at": "2024-01-01",
        "author": "Hongbo",
        "performance": {"acc": 0.78, "latency_p50": 2.1},
    },
    "v1.1": {
        "system": "You are an expert in construction safety analysis.",
        "user_template": (
            "Analyze this image based on the ontology:\n"
            "{ontology_yaml}\n\n"
            "Question: {question}\n"
            "Think step by step, then output JSON: {{\"reasoning\": ..., \"answer\": ..., \"confidence\": ...}}"
        ),
        "temperature": 0.0,
        "max_tokens": 800,
        "created_at": "2024-02-15",
        "author": "Hongbo",
        "performance": {"acc": 0.83, "latency_p50": 3.5},  # CoT 提升精度但变慢
        "changelog": "加了 step-by-step 推理 + confidence",
    },
    "v2.0_few_shot": {
        "system": "You are an expert in construction safety analysis.",
        "user_template": (
            "{few_shot_examples}\n\n"
            "Now analyze this image:\n{ontology_yaml}\n"
            "Question: {question}\nJSON output:"
        ),
        "temperature": 0.0,
        "max_tokens": 800,
        "few_shot_count": 3,
        "created_at": "2024-03-10",
        "performance": {"acc": 0.86, "latency_p50": 5.2},
        "changelog": "加 3 个 few-shot examples",
    },
}

# 模拟使用
def get_prompt(version, **kwargs):
    config = PROMPT_REGISTRY[version]
    user_msg = config["user_template"].format(**kwargs)
    return {
        "system": config["system"],
        "user": user_msg,
        "temperature": config["temperature"],
        "max_tokens": config["max_tokens"],
        "version": version,
    }


# 演示
prompt = get_prompt("v1.1",
                    ontology_yaml="entities:\n  - Worker\n  - Helmet",
                    question="Is the worker wearing PPE?")
print("Prompt v1.1:")
print("-" * 60)
print(f"System: {prompt['system'][:80]}...")
print(f"User:   {prompt['user'][:200]}...")
print(f"\n版本元数据: T={prompt['temperature']}, max_tokens={prompt['max_tokens']}")

# 性能对比
print("\n版本性能历史:")
print("=" * 60)
print(f"{'Version':<20s} {'Accuracy':>10s} {'P50 Latency':>12s} {'Note':<30s}")
print("-" * 60)
for v, cfg in PROMPT_REGISTRY.items():
    perf = cfg.get('performance', {})
    note = cfg.get('changelog', 'baseline')
    print(f"{v:<20s} {perf.get('acc', 0):>10.2f} {perf.get('latency_p50', 0):>10.1f}s   {note:<30s}")


In [ ]:
# 带缓存 + 重试 + 验证的 LLM 调用包装
import json
import hashlib
import time
from functools import wraps


class LLMClient:
    """工程级 LLM 客户端 (mock OpenAI API)"""
    
    def __init__(self):
        self.cache = {}
        self.call_log = []
    
    def _cache_key(self, prompt_dict):
        s = json.dumps(prompt_dict, sort_keys=True)
        return hashlib.md5(s.encode()).hexdigest()
    
    def call(self, prompt, retries=3, validate_json=True, schema=None):
        """
        prompt: dict from get_prompt()
        retries: 失败重试次数
        validate_json: 是否要求输出 JSON
        schema: 可选的 JSON schema 验证
        """
        # 缓存查询 (相同 prompt 直接返回)
        key = self._cache_key(prompt)
        if key in self.cache:
            self.call_log.append({'key': key[:8], 'cache_hit': True, 'time': 0})
            return self.cache[key]
        
        for attempt in range(retries):
            try:
                t0 = time.time()
                # 这里在真实环境是 openai.ChatCompletion.create(...)
                response = self._mock_llm_call(prompt)
                latency = time.time() - t0
                
                # JSON 验证
                if validate_json:
                    parsed = json.loads(response)
                    if schema:
                        self._validate_schema(parsed, schema)
                    response = parsed
                
                # 缓存 + 日志
                self.cache[key] = response
                self.call_log.append({
                    'key': key[:8], 'cache_hit': False, 
                    'time': latency, 'attempt': attempt + 1,
                    'version': prompt.get('version', 'unknown'),
                })
                return response
            
            except (json.JSONDecodeError, ValueError) as e:
                if attempt == retries - 1:
                    raise RuntimeError(f"Failed after {retries} retries: {e}")
                time.sleep(2 ** attempt * 0.1)  # 指数退避
    
    def _mock_llm_call(self, prompt):
        # 模拟 LLM 输出 (有时返回坏 JSON)
        import random
        if random.random() < 0.2:  # 20% 失败率
            return "This is not valid JSON"
        return json.dumps({
            "answer": "Yes, worker is wearing helmet",
            "reasoning": "Detected red helmet on worker's head",
            "confidence": 0.92,
        })
    
    def _validate_schema(self, obj, schema):
        for key in schema.get('required', []):
            if key not in obj:
                raise ValueError(f"Missing required field: {key}")
    
    def stats(self):
        n = len(self.call_log)
        cache_hits = sum(1 for c in self.call_log if c.get('cache_hit'))
        avg_attempts = sum(c.get('attempt', 1) for c in self.call_log if not c.get('cache_hit'))
        avg_attempts /= max(1, n - cache_hits)
        return {
            'total_calls': n,
            'cache_hit_rate': cache_hits / max(1, n),
            'avg_attempts_per_uncached': avg_attempts,
        }


# 演示
import random
random.seed(42)
client = LLMClient()

schema = {'required': ['answer', 'reasoning']}

print("模拟 10 次调用 (含 20% 失败率，自动重试):")
print("=" * 70)
for i in range(10):
    prompt = get_prompt("v1.1",
                        ontology_yaml=f"entities:\n  - Worker_{i % 3}",
                        question="Is the worker wearing PPE?")
    try:
        result = client.call(prompt, schema=schema)
        last = client.call_log[-1]
        marker = "📦cache" if last.get('cache_hit') else f"✅ retries={last.get('attempt')}"
        print(f"  call {i+1}: {marker}")
    except RuntimeError as e:
        print(f"  call {i+1}: ❌ {e}")

print("\n统计:")
for k, v in client.stats().items():
    print(f"  {k:30s} = {v}")


---
## 8. 评估 Pipeline 工程化 <a id='8'></a>

### 8.1 论文 vs 工程的评估区别

| 论文版 | 工程版 |
|---|---|
| 一次跑完出表格 | 持续运行，每次 push 都重跑 |
| 一个总 accuracy | per-class F1 + per-difficulty 拆分 |
| `python eval.py` | CLI: `eval --model v1 --testset v2 --output reports/` |
| 结果存 csv | 入 MLflow / wandb，可对比 |
| 失败 case 看一眼 | 自动收集到 error_analysis/ 目录 |

### 8.2 评估 Pipeline 设计


In [ ]:
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional


@dataclass
class EvalResult:
    """单次评估结果"""
    model_version: str
    prompt_version: str
    testset_version: str
    overall_acc: float
    per_class_f1: Dict[str, float]
    per_difficulty_acc: Dict[str, float]
    confusion_matrix: List[List[int]]
    error_cases: List[Dict] = field(default_factory=list)
    p50_latency: float = 0.0
    p99_latency: float = 0.0
    cost_per_query: float = 0.0  # API cost
    timestamp: str = ""


class EvaluationPipeline:
    """工程级评估管线"""
    
    def __init__(self, llm_client, prompt_version):
        self.client = llm_client
        self.prompt_version = prompt_version
    
    def run(self, testset, model_version="gpt-4"):
        """在 testset 上运行评估"""
        predictions = []
        latencies = []
        errors = []
        
        for item in testset:
            t0 = time.time()
            try:
                prompt = get_prompt(self.prompt_version, **item['prompt_kwargs'])
                pred = self.client.call(prompt)
                latency = time.time() - t0
                latencies.append(latency)
                predictions.append({
                    'id': item['id'],
                    'gt': item['ground_truth'],
                    'pred': pred,
                    'difficulty': item.get('difficulty', 'normal'),
                    'category': item['category'],
                })
                
                # 收集错误 case
                if pred.get('answer') != item['ground_truth']:
                    errors.append({
                        'id': item['id'],
                        'image': item.get('image_path'),
                        'gt': item['ground_truth'],
                        'pred': pred,
                        'category': item['category'],
                    })
            except Exception as e:
                errors.append({'id': item['id'], 'error': str(e)})
        
        # 计算指标
        return self._compute_metrics(predictions, errors, latencies, model_version)
    
    def _compute_metrics(self, predictions, errors, latencies, model_version):
        # Overall accuracy
        n_correct = sum(1 for p in predictions 
                        if p['pred'].get('answer') == p['gt'])
        overall = n_correct / max(1, len(predictions))
        
        # Per-class F1
        from collections import defaultdict
        cls_tp = defaultdict(int)
        cls_fp = defaultdict(int)
        cls_fn = defaultdict(int)
        for p in predictions:
            cat = p['category']
            if p['pred'].get('answer') == p['gt']:
                cls_tp[cat] += 1
            else:
                cls_fp[p['pred'].get('answer', 'unknown')] += 1
                cls_fn[cat] += 1
        per_class_f1 = {}
        for c in cls_tp:
            tp = cls_tp[c]; fp = cls_fp[c]; fn = cls_fn[c]
            p = tp / max(1, tp + fp); r = tp / max(1, tp + fn)
            f1 = 2 * p * r / max(1e-6, p + r)
            per_class_f1[c] = f1
        
        # Per-difficulty
        diff_groups = defaultdict(list)
        for p in predictions:
            d = p.get('difficulty', 'normal')
            diff_groups[d].append(p['pred'].get('answer') == p['gt'])
        per_diff_acc = {d: sum(v)/len(v) for d, v in diff_groups.items()}
        
        # Latency
        p50 = np.percentile(latencies, 50) if latencies else 0
        p99 = np.percentile(latencies, 99) if latencies else 0
        
        return EvalResult(
            model_version=model_version,
            prompt_version=self.prompt_version,
            testset_version='v1',
            overall_acc=overall,
            per_class_f1=per_class_f1,
            per_difficulty_acc=per_diff_acc,
            confusion_matrix=[],  # 简化省略
            error_cases=errors[:10],  # 保留前 10 个错误用于人工 review
            p50_latency=p50,
            p99_latency=p99,
            cost_per_query=0.01,  # mock
            timestamp=time.strftime('%Y-%m-%d %H:%M:%S'),
        )


# 演示
mock_testset = [
    {'id': i, 
     'prompt_kwargs': {'ontology_yaml': 'entities: Worker', 'question': f'q{i}'},
     'ground_truth': 'Yes, worker is wearing helmet',
     'category': ['ppe_check', 'fall_protection', 'crane_safety'][i % 3],
     'difficulty': ['easy', 'normal', 'hard'][i % 3],
     } for i in range(20)
]

random.seed(123)
pipeline = EvaluationPipeline(LLMClient(), prompt_version="v1.1")
result = pipeline.run(mock_testset)

print("评估结果:")
print("=" * 60)
print(f"  Model:    {result.model_version}")
print(f"  Prompt:   {result.prompt_version}")
print(f"  Overall:  {result.overall_acc:.3f}")
print(f"  P50 Lat:  {result.p50_latency:.2f}s")
print(f"  P99 Lat:  {result.p99_latency:.2f}s")
print(f"  Cost/q:   ${result.cost_per_query:.3f}")
print(f"\n  Per-class F1:")
for c, f in result.per_class_f1.items():
    print(f"    {c:20s}: {f:.3f}")
print(f"\n  Per-difficulty Acc:")
for d, a in result.per_difficulty_acc.items():
    print(f"    {d:10s}: {a:.3f}")
print(f"\n  Errors collected: {len(result.error_cases)} (for manual review)")


---
## 9. 面试 Talk Track 整理 <a id='9'></a>

### 9.1 「介绍一下你最得意的项目」(2 分钟版)

> 「我最近做了一个**本体引导的合成数据 Pipeline**，是把博士论文方法 (AEI/AiC TOP 期刊) 升级成工程级别。
> 
> 背景: 施工场景 AI 视觉训练数据严重不足 — 标注一张图要 30 分钟，长尾类别更难。
> 
> 方法: 用 OWL 本体约束 prompt 生成 → SDXL+ControlNet (MLSD 控制结构) 大规模合成 → CLIP Score 筛选 (τ=25 是消融出来的甜蜜点) → 与真实数据混合训练 ResNet/ViT。
> 
> 结果: ViT-B 在真实测试集 +X% acc，长尾类别 mAP 提升尤其明显。
> 
> 工程: 5 组件解耦 (ontology / generator / filter / dataset / train)，配置驱动 + Docker + GitHub。从博士的一次性实验升级为可复现的 pipeline，单条命令跑全流程。
> 
> 我准备好深入聊任何一个环节: 为什么 ControlNet 用零卷积，CLIP 筛选阈值怎么调，为什么混合比 30:70 在我的数据上最好。」

### 9.2 「你 Ontology Prompting 那篇文章具体怎么做的？」(3 分钟版)

> 「用本体约束 LLM 的推理过程。
> 
> **Why**: GPT-4 直接看图答施工安全题精度有限，因为它不知道你的领域规范。
> 
> **How**: 把规范知识编码成 OWL 本体 (entity / property / relation)，然后用模板把本体内容嵌入 prompt — 不是把整个本体扔进去，而是按问题类型抽相关子图。例如问 "worker 安全装备是否合规"，只抽 PPE 相关的 entity。
> 
> **关键工程细节**:
> - prompt 版本化 (v1.0 → v2.0_few_shot)，每版记录精度/延迟/成本
> - 输出 JSON Schema 强约束 + 失败重试
> - Temperature=0 + n=1 保证 deterministic
> - 评估分 per-class F1 + per-difficulty acc，不只看 overall
> - 错误 case 自动收集到 error_analysis/ 目录，每周 review 后回灌成 few-shot
> 
> **Failure mode 我清楚**: 本体覆盖外的场景退化为通用 prompt；图像质量极差时 LLM 拒答；occlusion 严重时空间推理失败。
> 
> **如果重做**: 加 RAG 把规范条文 retrieve 进来 (我下个月就要做的 GraphRAG 项目)，长期目标是替换 GPT-4 → QLoRA 微调 Qwen2-7B 私有化部署。」

### 9.3 题目对应 talk track 索引

| 题目 | 哪个项目 | 哪个 talk track | 几分钟 |
|---|---|---|---|
| 介绍最得意项目 | 合成数据 Pipeline | 9.1 | 2 |
| 介绍 LLM 经验 | Ontology Prompting | 9.2 | 3 |
| 介绍系统工程能力 | IoT 老年陪护 | (W1D7 已写) | 3 |
| 介绍图相关工作 | GNN 图纸审查 | (W3D7 准备) | 2 |
| 介绍 KG/RAG 能力 | GraphRAG (W6) | (W6D7 准备) | 3 |
| 介绍 Agent | 规范审查 Agent (W7) | (W7D7 准备) | 3 |
| 介绍系统工程 | 推理服务+MLOps (W9) | (W9D7 准备) | 3 |

到 W12 时你应该有 7-8 个滚瓜烂熟的 talk track。


---
## 10. W3 预习 + 下周准备 <a id='10'></a>

### 10.1 W3 七天预览

| Day | 主题 | 与本周连接 |
|---|---|---|
| W3D1 (周一) | GNN 原理 ★ | 你的另一个论文方向 (CAD 图纸审查) |
| W3D2 (周二) | GNN 进阶 + 图增强 ★ | 半监督 + 小数据训练 |
| W3D3 (周三) | 合成数据消融 + 终版 ★ | **D6 项目的完整版 + Docker + GitHub 发布** |
| W3D4 (周四) | LLM 架构深入 | MHA / MQA / GQA / RoPE / LLaMA |
| W3D5 (周五) | 知识工程 + Prompt Engineering | 复现你的 Ontology Prompting 论文 |
| W3D6 (周六) | P1 综合 + VLM | LLaVA / Whisper / 语音 demo |
| W3D7 (周日) | P1 总复盘 + 工程深挖③ | GNN 图纸项目深挖 |

### 10.2 W3 周末必须完成的事 (W3D3 锚点项目终版)

D3 是这周和下周的桥梁，必须完成:

- [ ] 把 D6 的 Mock 组件全部替换为真组件
  - `mock_generate` → `diffusers SDXL + ControlNet`
  - `mock_clip_score` → `clip.load("ViT-B/32")`
- [ ] 跑完整的 5×5 消融 (50 组实验)
- [ ] 写 README.md (架构图 + 一键运行 + 实验结果表)
- [ ] Dockerfile + docker-compose.yml
- [ ] GitHub 推送 → 在简历放 link

### 10.3 W2 自检清单

理论:
- [ ] 能闭眼画 Transformer 完整计算图 (含 shape)
- [ ] 能 1 分钟讲清 KV Cache 原理 + 显存计算
- [ ] 能 1 分钟讲清 CLIP InfoNCE + 双塔设计
- [ ] 能 2 分钟讲清 DDPM → DDIM → CFG → ControlNet 演进
- [ ] 能 1 分钟讲清你合成数据的 domain gap 解决方案

工程:
- [ ] D6 Pipeline 跑通 + 4×2 矩阵有结果
- [ ] 消融运行器骨架可用 (D3 直接填充)
- [ ] 配置 + 日志基础设施就位

talk track:
- [ ] 「最得意项目」 (合成数据 Pipeline) 2 分钟版
- [ ] Ontology Prompting 3 分钟版
- [ ] 准备好被追问 5 层 why

### 10.4 工程深挖② 收尾

今天我们做了:
1. ✅ Transformer 完整计算图 + Pre-LN 反向梯度
2. ✅ CLIP 对比学习数学根基
3. ✅ 5+5 个脱稿 QA 模板
4. ✅ Ontology Prompting 5 层 why 工程深挖
5. ✅ Prompt 版本管理 + LLM 客户端工程化
6. ✅ 评估 Pipeline 设计 (per-class F1 / per-difficulty / 错误收集)
7. ✅ Talk track 索引

**核心收获**: 你的博士工作不是「做完了的论文」而是「面试时的故事弹药」—但要把工程层的 5 层 why 想清楚，论文里没写的细节才是面试官追问的题。


In [ ]:
print("=" * 60)
print("W2 完整结束! 🎉🎉🎉")
print("=" * 60)
print("""
P1 阶段进度: W1 ✅ + W2 ✅ + W3 (本周开始)

W2 沉淀清单:
  ✅ Transformer / CLIP / Diffusion 全栈打通
  ✅ nanoGPT 手写 + KV Cache + 5 种采样
  ✅ 合成数据 Pipeline v1 (锚点项目骨架)
  ✅ 10+ 脱稿 QA 模板
  ✅ Ontology Prompting 工程化复盘
  ✅ Prompt 版本管理 + 评估 Pipeline 设计
  ✅ 2 个 talk track (合成数据 + Ontology Prompting)

W3 第一目标: 周三 (D3) 把 D6 的合成数据 Pipeline 升级到 GitHub 可发布版
  - 真 SDXL + ControlNet 替换 mock
  - 真 CLIP 替换 mock CLIP Score
  - 完整 5×5 消融
  - Dockerfile + README + 一键运行

下周再见，你状态在线! 💪
""")